# RecurQuant: packed Qwen3.5 recurrent-state demo

Run these cells from top to bottom in a fresh **Google Colab GPU runtime**. The demo installs RecurQuant from an immutable GitHub ref, downloads the public `Qwen/Qwen3.5-0.8B-Base` checkpoint at the revision pinned by RecurQuant, and runs the installed `recurquant qwen35` command. No Hugging Face token is required.

The command uses RecurQuant's `mixed-v02` default: model layer 0 is stored at INT8 and the other supported recurrent layers at INT4, with group size 128 and FP16 scales. This is an alpha research demo, not a speed or whole-model memory benchmark.

## 1. Install the pinned package

The immutable `v0.2.0a1` tag keeps the notebook tied to the tested alpha release.

In [ ]:
import subprocess
import sys

RECURQUANT_REPOSITORY = "https://github.com/Labeeb2339/recurquant.git"
RECURQUANT_REF = "v0.2.0a1"  # Immutable alpha release tag.
install_target = f"recurquant @ git+{RECURQUANT_REPOSITORY}@{RECURQUANT_REF}"
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--disable-pip-version-check",
        install_target,
    ],
    check=True,
)


## 2. Check the Colab runtime

If CUDA is missing, choose **Runtime > Change runtime type > GPU** and run the notebook again. BF16 is the configuration used for RecurQuant's public full-model fidelity evidence. On an older GPU without BF16, the installed command can fall back to FP16, but that weight dtype is outside the current full-model evidence boundary.

In [ ]:
import warnings
from importlib.metadata import version

import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is unavailable. In Colab, select Runtime > Change runtime type > GPU, "
        "then run all cells again."
    )

gpu_name = torch.cuda.get_device_name(0)
bf16_supported = torch.cuda.is_bf16_supported()
print(f"GPU: {gpu_name}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"BF16 supported: {bf16_supported}")
print(f"RecurQuant: {version('recurquant')}")
print(f"PyTorch: {torch.__version__}")
print(f"Transformers: {version('transformers')}")

if not bf16_supported:
    warnings.warn(
        "This GPU does not support BF16. The CLI will fall back to FP16, but "
        "RecurQuant's public full-model fidelity evidence currently covers BF16 only.",
        RuntimeWarning,
        stacklevel=1,
    )


## 3. Run the installed mixed-v02 default

The first run downloads the pinned 0.8B model and can take several minutes. The command deliberately omits `--policy`, so it exercises the installed `mixed-v02` default rather than the uniform-INT4 stress baseline.

In [ ]:
import shlex

PROMPT = "Explain recurrent-state quantization in two short sentences."
MAX_NEW_TOKENS = 24
command = [
    "recurquant",
    "qwen35",
    "--device",
    "cuda",
    "--max-new-tokens",
    str(MAX_NEW_TOKENS),
    "--prompt",
    PROMPT,
]
print("Running:", shlex.join(command))
subprocess.run(command, check=True)


## Read the byte counters honestly

- `resident_recurrent_state_bytes` is the exact live tensor storage used by the packed persistent recurrent states.
- `full_precision_equivalent_recurrent_state_bytes` is how many bytes those same states would occupy in their original model dtype.
- `largest_materialized_recurrent_state_bytes` is the size of the largest single state materialized while a layer runs. It is **not** the layer's full workspace or a CUDA allocator peak.
- `resident_compression_ratio` compares only the first two counters.

These numbers do not measure total GPU memory, model weights, ordinary attention KV caches, activations, temporary quantization workspaces, or inference speed. The current Python implementation materializes one recurrent state during layer execution, so this notebook makes no claim of lower peak VRAM or lower latency. The generated text is a functional demo, not a generated-output quality evaluation.